In [29]:
import psycopg2

POSTGRES_HOST: str = "localhost"
POSTGRES_PORT: int = 5432
POSTGRES_DBNAME: str = "dms_meta"
POSTGRES_USER: str = "dms"
POSTGRES_PASSWORD: str = "dms"

pg_conn = psycopg2.connect(
    host=POSTGRES_HOST,
    port=POSTGRES_PORT,
    database=POSTGRES_DBNAME,
    user=POSTGRES_USER,
    password=POSTGRES_PASSWORD,
)

In [30]:
from pathlib import Path
current_directory = Path.cwd()

In [31]:
from src.dms.adapters import AzureBlobStorageClient, PostgresMetadataRepository
from src.dms.service import DmsService
import os

In [32]:
from azure.storage.blob import BlobServiceClient
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

In [33]:
storage_client = AzureBlobStorageClient(blob_service_client)
metadata_repo = PostgresMetadataRepository(pg_conn)

dms_service = DmsService(storage_client=storage_client, metadata_repository=metadata_repo)

In [34]:
from src.async_processing import AsyncDocumentProcessor
async_processor = AsyncDocumentProcessor()

In [35]:
from datetime import datetime

# Resolve sample file
sample_pdf_path = current_directory / "data" / "AlliedEsportsEntertainmentInc_20190815_8-K_EX-10.19_11788293_EX-10.19_Content License Agreement.pdf"
assert sample_pdf_path.exists(), "Sample PDF not found"

DOCUMENT_TYPE: str = "license-agreement"

document_id = dms_service.upload_document(
    file_path=sample_pdf_path,
    document_type=DOCUMENT_TYPE,
    source_filename=sample_pdf_path.name,
)

print("Uploaded:", document_id)

metadata = dms_service.get_document(document_id=document_id)
print("Metadata keys:", sorted(list(metadata.keys())) if metadata else None)

# List jobs
jobs = dms_service.get_extraction_jobs(document_id=document_id)
print(f"\nExtraction jobs created: {len(jobs)}")
for job in jobs:
    print(f"- Job ID: {job['id']}")
    print(f"  Status: {job['status']}")
    print(f"  Created: {job['created_at']}")



Uploaded: 122945f7-2b1c-4997-bab6-94eb88499b88
Metadata keys: ['acu_result_blob_path', 'blob_path', 'created_at', 'document_type', 'file_size', 'hash_sha256', 'id', 'linked_entity', 'linked_entity_id', 'mime_type', 'processing_status', 'source_filename', 'text_extraction_status', 'updated_at']

Extraction jobs created: 1
- Job ID: 03751339-9e34-42e4-b538-7f4736a2878c
  Status: pending
  Created: 2026-02-13 06:41:35.451679+00:00


In [36]:
# Trigger async processing
print("Triggering async processing...")
task_id = async_processor.trigger_processing(document_id=document_id)

if task_id:
    print(f"Async processing started with task ID: {task_id}")
    print("Processing is now running in the background Celery worker")
else:
    print("Failed to start async processing")

Triggering async processing...
Async processing started with task ID: dfb1479f-e929-402f-be84-7d5fad9520fb
Processing is now running in the background Celery worker


In [37]:
import time
# Monitor processing status
print("Monitoring processing status...")
print("This may take a few minutes as the Celery worker processes the document")

for i in range(20):  # Monitor for up to 20 iterations
    status = async_processor.get_processing_status(document_id=document_id)
    
    print(f"\n--- Status Check {i+1} ---")
    print(f"Text extraction status: {status.get('text_extraction_status')}")
    print(f"Processing status: {status.get('processing_status')}")
    
    # Check extraction jobs
    jobs = status.get('extraction_jobs', [])
    if jobs:
        latest_job = jobs[0]
        print(f"Latest job status: {latest_job.get('status')}")
        if latest_job.get('error_message'):
            print(f"Error: {latest_job.get('error_message')}")
    
    # Check if processing is complete
    if status.get('processing_status') == 'done':
        print("\n✓ Processing completed successfully!")
        break
    elif status.get('processing_status') == 'failed':
        print("\n✗ Processing failed")
        break
    
    # Wait before next check
    time.sleep(7)
else:
    print("\nMonitoring timeout - processing may still be running")

Monitoring processing status...
This may take a few minutes as the Celery worker processes the document

--- Status Check 1 ---
Text extraction status: in progress
Processing status: acu running
Latest job status: pending

--- Status Check 2 ---
Text extraction status: in progress
Processing status: acu running
Latest job status: pending

--- Status Check 3 ---
Text extraction status: in progress
Processing status: acu running
Latest job status: pending

--- Status Check 4 ---
Text extraction status: in progress
Processing status: acu running
Latest job status: pending

--- Status Check 5 ---
Text extraction status: in progress
Processing status: acu running
Latest job status: pending

--- Status Check 6 ---
Text extraction status: in progress
Processing status: acu running
Latest job status: pending

--- Status Check 7 ---
Text extraction status: in progress
Processing status: acu running
Latest job status: pending

--- Status Check 8 ---
Text extraction status: in progress
Processing